In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 219 (delta 4), reused 9 (delta 2), pack-reused 202 (from 1)
Receiving objects: 100% (219/219), 920.99 KiB | 13.75 MiB/s, done.
Resolving deltas: 100% (131/131), done.
/content/InkubaLM-Challenge


In [2]:

!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system =

In [4]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)
from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except:
    os.environ["HF_TOKEN"] = "----"

login(token=os.environ["HF_TOKEN"])

token = os.environ["HF_TOKEN"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Load Base Model and Run Inference on Test Set for Baseline Performance

In [11]:
# Load base model
base_model = AutoModelForCausalLM.from_pretrained("lelapa/InkubaLM-0.4B", device_map="auto")
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B")

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/991k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.95M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [12]:
print("# Loading datasets")
train_dataset = data_utils.load_and_combine_datasets("Train")
test_dataset = data_utils.load_and_combine_datasets("Test")

# Loading datasets


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/39.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/400 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

Common Columns: ['ID', 'langs', 'targets', 'instruction', 'inputs']


Casting the dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

In [29]:
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance"
os.makedirs(output_path, exist_ok=True)

In [32]:
BASE_PROMPT = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n ### Instruction: {}\n\n ### Response: "

In [31]:
from utils import ab_testing

ab_testing.run_inference_on_tasks(
    output_path=output_path,
    lora_model=base_model,
    tokenizer=tokenizer,
    new_model_function=new_model_function,
    sample_size=50
)

NameError: name 'ab_inference' is not defined

In [13]:
results_df = inference.apply_inference_to_test_data(base_model, tokenizer, test_dataset)
results_df.to_csv("submission_test_baseline_performance.csv", index=False)

Generating Responses:   0%|          | 0/900 [00:00<?, ?it/s]

In [20]:
results_df.head()

,ID,langs,instruction,inputs,targets,generated,Response
0,ID_f3c74c7b_sentiment_test__hausa,hausa,Gano ra'ayin da aka bayyana a cikin wannan rub...,@user ynxu fha da kanada kudi shikenan duk kay...,None,"#neeraplāsā, #yace da #aminalāsā,",0
1,ID_aad19dbf_sentiment_test__hausa,hausa,Za ka iya tantance yanayin wannan rubutu? Bi w...,@user alhamdulillah babu abinda zamuce sai god...,None,#adawa #addu'aram #nafarko #abuja #abinda,0
2,ID_f6de0381_sentiment_test__hausa,hausa,Za ka iya tantance yanayin wannan rubutu? Bi w...,@user ke ina ruwan ki 😬 ba harkar film bane ba,None,#aikace-aikacen Yanayi a kan layi.osa (4yar ta,0
3,ID_cbec84fe_sentiment_test__swahili,swahili,Changanua mawazo ya matini yanayofuata na uain...,matokeo chanya liverais magufuli katika uzindu...,None,"""Ranka ni la kifaya (tafsir) na madheemah",0
4,ID_885caf5c_sentiment_test__hausa,hausa,Tantance ra’ayin wannan rubutu kuma a rarraba ...,@user 🤣 akwai lauje cikin nadi gaskiya.,None,@user ☀️ ẽshi☀️😀,0


In [28]:
results_df

,ID,langs,instruction,inputs,targets,generated,Response
0,ID_f3c74c7b_sentiment_test__hausa,hausa,Gano ra'ayin da aka bayyana a cikin wannan rub...,@user ynxu fha da kanada kudi shikenan duk kay...,None,"#neeraplāsā, #yace da #aminalāsā,",0
1,ID_aad19dbf_sentiment_test__hausa,hausa,Za ka iya tantance yanayin wannan rubutu? Bi w...,@user alhamdulillah babu abinda zamuce sai god...,None,#adawa #addu'aram #nafarko #abuja #abinda,0
2,ID_f6de0381_sentiment_test__hausa,hausa,Za ka iya tantance yanayin wannan rubutu? Bi w...,@user ke ina ruwan ki 😬 ba harkar film bane ba,None,#aikace-aikacen Yanayi a kan layi.osa (4yar ta,0
3,ID_cbec84fe_sentiment_test__swahili,swahili,Changanua mawazo ya matini yanayofuata na uain...,matokeo chanya liverais magufuli katika uzindu...,None,"""Ranka ni la kifaya (tafsir) na madheemah",0
4,ID_885caf5c_sentiment_test__hausa,hausa,Tantance ra’ayin wannan rubutu kuma a rarraba ...,@user 🤣 akwai lauje cikin nadi gaskiya.,None,@user ☀️ ẽshi☀️😀,0
...,...,...,...,...,...,...,...
895,ID_88cd08fe_test__afrixnli_hau,hau,"Is the following question True, False or Neither?",Bradley ya fito daga Missouri.,None,RSS.osaqon Caris : Is the following questionus...,0
896,ID_404bc9af_test__afrixnli_swa,swa,"Is the following question True, False or Neither?",Brahma ni sehemu muhimu zaidi ya utatu.,None,#Dankali #Amarqar Usama: https://m.facebook.com/,0
897,ID_a399ed32_test__afrixnli_swa,swa,"Is the following question True, False or Neither?",Ilinichukua muda wa miezi 13 kukiweka kichemsh...,None,Ashekuka kicin kinata na kicini-kifo.ibu,0
898,ID_9f28a032_test__afrixnli_hau,hau,"Is the following question True, False or Neither?",an hada da nazarin Marxist-Leninist a harkar k...,None,Fim ɗin kuma yana nufin ba kawai suna bayyana ...,0
